## LLM Evaluation

In [ ]:
#pip install evaluate

In [2]:
!python -m pip install evaluate rouge_score --quiet


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
prediction = ["Today is Wednesday and I am enjoying it here. i want".lower()]
actual = ["Today is wednesday and I want to travel somwhere.".lower()]

## BLEU (Bilingual Evaluation Understudy)

**Definition**: BLEU is a metric for evaluating a generated sentence to a reference sentence. It measures the n-gram precision with a penalty for overly short sentences.

**Interpretation**:
- **High BLEU Score**: Indicates good performance (e.g., scores above 0.5 or 50%).
- **Low BLEU Score**: Indicates poor performance.

**Benchmark**: For machine translation tasks, a BLEU score above 0.3 (30%) is considered reasonable, while scores above 0.5 (50%) are considered good.

---



In [4]:
import evaluate
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=prediction,references=actual)
print(results)

{'bleu': 0.0, 'precisions': [1.0, 1.0, 1.0, 0.0], 'brevity_penalty': 0.09697196786440505, 'length_ratio': 0.3, 'translation_length': 3, 'reference_length': 10}


## ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

**Definition**: ROUGE measures the overlap of n-grams between the generated sentence and the reference sentence, focusing on recall.

**Types**:
- **ROUGE-1**: Measures the overlap of unigrams.
- **ROUGE-2**: Measures the overlap of bigrams.
- **ROUGE-L**: Measures the longest common subsequence.

**Interpretation**:
- **High ROUGE Score**: Indicates good performance.
- **Low ROUGE Score**: Indicates poor performance.

**Benchmark**: For summarization tasks, ROUGE scores of 0.5 (50%) or higher are considered good.

---



In [5]:
rouge = evaluate.load("rouge")
results = rouge.compute(predictions=prediction,references=actual)
print(results)

{'rouge1': np.float64(0.5), 'rouge2': np.float64(0.4), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}


In [ ]:
meteor = evaluate.load("meteor")
results = meteor.compute(predictions=prediction,references=actual)
print(results)

# LLM Evaluation with MLflow Example Notebook


In [7]:
!python -m pip install mlflow --quiet


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [8]:
import pandas as pd
import mlflow

## Basic Question-Answering Evaluation

Create a test case of `inputs` that will be passed into the model and `ground_truth` which will be used to compare against the generated output from the model.

In [9]:
eval_df = pd.DataFrame(
    {
        "inputs": [
            "How does useEffect() work?",
            "What does the static keyword in a function mean?",
            "What does the 'finally' block in Python do?",
            "What is the difference between multiprocessing and multithreading?",
        ],
        "ground_truth": [
            "The useEffect() hook tells React that your component needs to do something after render. React will remember the function you passed (we’ll refer to it as our “effect”), and call it later after performing the DOM updates.",
            "Static members belongs to the class, rather than a specific instance. This means that only one instance of a static member exists, even if you create multiple objects of the class, or if you don't create any. It will be shared by all objects.",
            "'Finally' defines a block of code to run when the try... except...else block is final. The finally block will be executed no matter if the try block raises an error or not.",
            "Multithreading refers to the ability of a processor to execute multiple threads concurrently, where each thread runs a process. Whereas multiprocessing refers to the ability of a system to run multiple processors in parallel, where each processor can run one or more threads.",
        ],
    }
)
eval_df

,inputs,ground_truth
0,How does useEffect() work?,The useEffect() hook tells React that your com...
1,What does the static keyword in a function mean?,"Static members belongs to the class, rather th..."
2,What does the 'finally' block in Python do?,'Finally' defines a block of code to run when ...
3,What is the difference between multiprocessing...,Multithreading refers to the ability of a proc...


In [10]:
from google import genai
client = genai.Client()

In [12]:
import mlflow
mlflow.set_tracking_uri("http://20.75.92.162:5000/")

In [13]:
mlflow.set_experiment("LLM Evaluation")

2025/10/08 13:02:19 INFO mlflow.tracking.fluent: Experiment with name 'LLM Evaluation' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/785876888742354870', creation_time=1759908740040, experiment_id='785876888742354870', last_update_time=1759908740040, lifecycle_stage='active', name='LLM Evaluation', tags={}>

In [15]:
def predict(data: pd.DataFrame) -> list[str]:
    predictions = []
    for _, row in data.iterrows():
        # Fill in variables in the prompt template
        content="Answer the following question " + row['inputs']

        completion = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=content,
        )
        predictions.append(completion.text)

    return predictions

In [18]:
!python -m pip install transformers textstat tiktoken --quiet


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [19]:
with mlflow.start_run() as run:
    
    results = mlflow.evaluate(
        model=predict,
        data=eval_df,
        targets="ground_truth",  # specify which column corresponds to the expected output
        model_type="question-answering",  # model type indicates which metrics are relevant for this task
        evaluators="default",
        extra_metrics=[mlflow.metrics.latency(),
                       mlflow.metrics.flesch_kincaid_grade_level(),
                       mlflow.metrics.ari_grade_level(),
                       mlflow.metrics.rougeL(),mlflow.metrics.bleu()]
    )
results.metrics

2025/10/08 13:10:51 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
2025/10/08 13:11:31 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2025/10/08 13:11:37 WARNING mlflow.metrics.metric_definitions: Failed to load 'toxicity' metric (error: NameError("name 'torch' is not defined")), skipping metric logging.
2025/10/08 13:11:37 WARNING mlflow.models.evaluation.utils.metric: Did not log metric 'toxicity' at index 1 in the `extra_metrics` parameter because it returned None.
2025/10/08 13:11:40 WARNING mlflow.metrics.metric_definitions: Failed to load 'toxicity' metric (error: NameError("name 'torch' is not defined"

🏃 View run respected-whale-705 at: http://20.75.92.162:5000/#/experiments/785876888742354870/runs/4a185e92c46a4357b1ab033ee383d3d7
🧪 View experiment at: http://20.75.92.162:5000/#/experiments/785876888742354870


{'latency/mean': np.float64(9.977612435817719),
 'latency/variance': np.float64(3.8428251897028183),
 'latency/p90': np.float64(11.547556710243224),
 'flesch_kincaid_grade_level/v1/mean': np.float64(12.709514966512813),
 'flesch_kincaid_grade_level/v1/variance': np.float64(1.4852180668125314),
 'flesch_kincaid_grade_level/v1/p90': np.float64(13.992650739441022),
 'ari_grade_level/v1/mean': np.float64(14.132969799467263),
 'ari_grade_level/v1/variance': np.float64(3.2634668984743325),
 'ari_grade_level/v1/p90': np.float64(15.818334473443267),
 'exact_match/v1': 0.0,
 'rougeL/v1/mean': np.float64(0.06201776517670971),
 'rougeL/v1/variance': np.float64(0.0002184721499458014),
 'rougeL/v1/p90': np.float64(0.07783991585971783),
 'bleu/v1/mean': np.float64(0.001933395109070272),
 'bleu/v1/variance': np.float64(1.1214049943330548e-05),
 'bleu/v1/p90': np.float64(0.005413506305396763)}

[Trace(trace_id=tr-42b603d649c792ed1671e9e0980197d7), Trace(trace_id=tr-57cd1f84c7d252650b44183a47c7e75f), Trace(trace_id=tr-fd50b1ac7a6ea1cb775e15f72e40373f), Trace(trace_id=tr-9920f69254799655c3a87229d0a69bca)]

Inspect the evaluation results table as a dataframe to see row-by-row metrics to further assess model performance

In [20]:
results.tables["eval_results_table"]

,inputs,ground_truth,outputs,latency,token_count,flesch_kincaid_grade_level/v1/score,ari_grade_level/v1/score,rougeL/v1/score,bleu/v1/score
0,How does useEffect() work?,The useEffect() hook tells React that your com...,`useEffect()` is a fundamental React Hook that...,11.632714,1237,11.094189,11.355303,0.055825,0.000000
1,What does the static keyword in a function mean?,"Static members belongs to the class, rather th...",The `static` keyword in a function declaration...,11.348856,1310,12.456509,14.336827,0.048951,0.007734
2,What does the 'finally' block in Python do?,'Finally' defines a block of code to run when ...,The `finally` block in Python is used in conju...,6.707703,739,12.771256,14.423725,0.087129,0.000000
3,What is the difference between multiprocessing...,Multithreading refers to the ability of a proc...,The key difference between multiprocessing and...,10.221177,1117,14.516105,16.416024,0.056166,0.000000


## LLM-judged correctness with Gemini

Construct an answer similarity metric using the `answer_similarity()` metric factory function.

In [28]:
from mlflow.metrics.genai import EvaluationExample, answer_similarity

# Create an example to describe what answer_similarity means like for this problem.
example = EvaluationExample(
    input="What is MLflow?",
    output="MLflow is an open-source platform for managing machine "
    "learning workflows, including experiment tracking, model packaging, "
    "versioning, and deployment, simplifying the ML lifecycle.",
    score=4,
    justification="The definition effectively explains what MLflow is "
    "its purpose, and its developer. It could be more concise for a 5-score.",
    grading_context={
        "targets": "MLflow is an open-source platform for managing "
        "the end-to-end machine learning (ML) lifecycle. It was developed by Databricks, "
        "a company that specializes in big data and machine learning solutions. MLflow is "
        "designed to address the challenges that data scientists and machine learning "
        "engineers face when developing, training, and deploying machine learning models."
    },
)

# Construct the metric using OpenAI GPT-4 as the judge
answer_similarity_metric = answer_similarity(model="gemini:/gemini-2.0-flash", examples=[example])

print(answer_similarity_metric)

EvaluationMetric(name=answer_similarity, greater_is_better=True, long_name=answer_similarity, version=v1, metric_details=
Task:
You must return the following fields in your response in two lines, one below the other:
score: Your numerical score for the model's answer_similarity based on the rubric
justification: Your reasoning about the model's answer_similarity score

You are an impartial judge. You will be given an input that was sent to a machine
learning model, and you will be given an output that the model produced. You
may also be given additional information that was used by the model to generate the output.

Your task is to determine a numerical score called answer_similarity based on the input and output.
A definition of answer_similarity and a grading rubric are provided below.
You must use the grading rubric to determine your score. You must also justify your score.

Examples could be included below for reference. Make sure to use them as references and to
understand them be

Call `mlflow.evaluate()` again but with your new `answer_similarity_metric`

In [29]:
with mlflow.start_run() as run:
    results = mlflow.evaluate(
        model=predict,
        data=eval_df,
        targets="ground_truth",
        model_type="question-answering",
        evaluators="default",
        extra_metrics=[answer_similarity_metric],  # use the answer similarity metric created above
    )
results.metrics

2025/10/08 13:22:44 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
2025/10/08 13:23:25 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2025/10/08 13:23:26 WARNING mlflow.metrics.metric_definitions: Failed to load 'toxicity' metric (error: NameError("name 'torch' is not defined")), skipping metric logging.
2025/10/08 13:23:26 WARNING mlflow.models.evaluation.utils.metric: Did not log metric 'toxicity' at index 1 in the `extra_metrics` parameter because it returned None.
100%|██████████| 1/1 [00:00<00:00, 9383.23it/s]
/home/zadmin/Desktop/test/GAAI-B5-GCP/genai/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/zadmin/Desktop/test/GAAI-B5-GCP/genai/lib/python3.11/site-packages/numpy/_core/_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/zadmin/Desktop/

🏃 View run grandiose-shoat-852 at: http://20.75.92.162:5000/#/experiments/785876888742354870/runs/a4977f1a99da42ceb44704c3bb8909fc
🧪 View experiment at: http://20.75.92.162:5000/#/experiments/785876888742354870


{'flesch_kincaid_grade_level/v1/mean': np.float64(11.974080931799067),
 'flesch_kincaid_grade_level/v1/variance': np.float64(0.7027663534461486),
 'flesch_kincaid_grade_level/v1/p90': np.float64(12.89256391646586),
 'ari_grade_level/v1/mean': np.float64(13.692477929458255),
 'ari_grade_level/v1/variance': np.float64(1.0813315224059057),
 'ari_grade_level/v1/p90': np.float64(14.62377184390547),
 'exact_match/v1': 0.0,
 'answer_similarity/v1/mean': np.float64(nan),
 'answer_similarity/v1/variance': np.float64(nan)}

[Trace(trace_id=tr-7174c7e007939884e9c7a81dc3c3436b), Trace(trace_id=tr-476f8c5aaf704c6b725bf4731ff5c96a), Trace(trace_id=tr-1dd151de616ff22e92955d8f4d884909), Trace(trace_id=tr-81ab2ee5ed3f25a6ac2438f9f987511c)]

See the row-by-row LLM-judged answer similarity score and justifications

In [30]:
results.tables["eval_results_table"]

,inputs,ground_truth,outputs,token_count,flesch_kincaid_grade_level/v1/score,ari_grade_level/v1/score,answer_similarity/v1/score,answer_similarity/v1/justification
0,How does useEffect() work?,The useEffect() hook tells React that your com...,`useEffect()` is a React Hook that lets you pe...,1335,10.904567,12.032328,NaN,Failed to score model on payload. Error: Provi...
1,What does the static keyword in a function mean?,"Static members belongs to the class, rather th...",The `static` keyword in a function declaration...,1184,12.193917,13.859546,NaN,Failed to score model on payload. Error: Provi...
2,What does the 'finally' block in Python do?,'Finally' defines a block of code to run when ...,The `finally` block in Python is used in conju...,941,11.605856,13.977135,NaN,Failed to score model on payload. Error: Provi...
3,What is the difference between multiprocessing...,Multithreading refers to the ability of a proc...,The key difference between multiprocessing and...,1157,13.191984,14.900902,NaN,Failed to score model on payload. Error: Provi...


## Custom LLM-judged metric for professionalism

Create a custom metric that will be used to determine professionalism of the model outputs. Use `make_genai_metric` with a metric definition, grading prompt, grading example, and judge model configuration

In [31]:
from mlflow.metrics.genai import EvaluationExample, make_genai_metric

professionalism_metric = make_genai_metric(
    name="professionalism",
    definition=(
        "Professionalism refers to the use of a formal, respectful, and appropriate style of communication that is tailored to the context and audience. It often involves avoiding overly casual language, slang, or colloquialisms, and instead using clear, concise, and respectful language"
    ),
    grading_prompt=(
        "Professionalism: If the answer is written using a professional tone, below "
        "are the details for different scores: "
        "- Score 1: Language is extremely casual, informal, and may include slang or colloquialisms. Not suitable for professional contexts."
        "- Score 2: Language is casual but generally respectful and avoids strong informality or slang. Acceptable in some informal professional settings."
        "- Score 3: Language is balanced and avoids extreme informality or formality. Suitable for most professional contexts. "
        "- Score 4: Language is noticeably formal, respectful, and avoids casual elements. Appropriate for business or academic settings. "
        "- Score 5: Language is excessively formal, respectful, and avoids casual elements. Appropriate for the most formal settings such as textbooks. "
    ),
    examples=[
        EvaluationExample(
            input="What is MLflow?",
            output=(
                "MLflow is like your friendly neighborhood toolkit for managing your machine learning projects. It helps you track experiments, package your code and models, and collaborate with your team, making the whole ML workflow smoother. It's like your Swiss Army knife for machine learning!"
            ),
            score=2,
            justification=(
                "The response is written in a casual tone. It uses contractions, filler words such as 'like', and exclamation points, which make it sound less professional. "
            ),
        )
    ],
    version="v1",
    model="gemini:/gemini-2.0-flash",
    parameters={"temperature": 0.0},
    grading_context_columns=[],
    aggregations=["mean", "variance", "p90"],
    greater_is_better=True,
)

print(professionalism_metric)

EvaluationMetric(name=professionalism, greater_is_better=True, long_name=professionalism, version=v1, metric_details=
Task:
You must return the following fields in your response in two lines, one below the other:
score: Your numerical score for the model's professionalism based on the rubric
justification: Your reasoning about the model's professionalism score

You are an impartial judge. You will be given an input that was sent to a machine
learning model, and you will be given an output that the model produced. You
may also be given additional information that was used by the model to generate the output.

Your task is to determine a numerical score called professionalism based on the input and output.
A definition of professionalism and a grading rubric are provided below.
You must use the grading rubric to determine your score. You must also justify your score.

Examples could be included below for reference. Make sure to use them as references and to
understand them before complet

Call `mlflow.evaluate` with your new professionalism metric. 

In [32]:
with mlflow.start_run() as run:
    results = mlflow.evaluate(
        model=predict,
        data=eval_df,
        model_type="question-answering",
        evaluators="default",
        extra_metrics=[professionalism_metric],  # use the professionalism metric we created above
    )
print(results.metrics)

2025/10/08 13:24:50 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
2025/10/08 13:25:28 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2025/10/08 13:25:29 WARNING mlflow.metrics.metric_definitions: Failed to load 'toxicity' metric (error: NameError("name 'torch' is not defined")), skipping metric logging.
2025/10/08 13:25:29 WARNING mlflow.models.evaluation.utils.metric: Did not log metric 'toxicity' at index 1 in the `extra_metrics` parameter because it returned None.
2025/10/08 13:25:29 WARNING mlflow.models.evaluation.utils.metric: Did not log metric 'exact_match' at index 4 in the `extra_metrics` parameter because it returned None.
100%|██████████| 1/1 [00:00<00:00, 10512.04it/s]
/home/zadmin/Desktop/test/GAAI-B5-GCP/genai/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/zadmin/Desktop/test/GAAI-B5-GCP/genai/lib/

🏃 View run bouncy-bug-109 at: http://20.75.92.162:5000/#/experiments/785876888742354870/runs/7ebc2bbf116c49f09cf40e49975bc105
🧪 View experiment at: http://20.75.92.162:5000/#/experiments/785876888742354870
{'flesch_kincaid_grade_level/v1/mean': np.float64(11.487463613756521), 'flesch_kincaid_grade_level/v1/variance': np.float64(1.0219241772916945), 'flesch_kincaid_grade_level/v1/p90': np.float64(12.400308098745944), 'ari_grade_level/v1/mean': np.float64(13.19510481085545), 'ari_grade_level/v1/variance': np.float64(1.6542296760946367), 'ari_grade_level/v1/p90': np.float64(14.47770266335365), 'professionalism/v1/mean': np.float64(nan), 'professionalism/v1/variance': np.float64(nan)}


[Trace(trace_id=tr-fc82b4f85518c9214f0f400c7398c135), Trace(trace_id=tr-919856ed3e9280d357765a3b7c9a8f2f), Trace(trace_id=tr-49f05cd8f1716c6b27df6f9ccc5fcaf3), Trace(trace_id=tr-507596682bbd0dd4273956ec0a919965)]

In [33]:
results.tables["eval_results_table"]

,inputs,ground_truth,outputs,token_count,flesch_kincaid_grade_level/v1/score,ari_grade_level/v1/score,professionalism/v1/score,professionalism/v1/justification
0,How does useEffect() work?,The useEffect() hook tells React that your com...,`useEffect()` in React is a powerful hook that...,1899,11.653976,12.640378,NaN,Failed to score model on payload. Error: Provi...
1,What does the static keyword in a function mean?,"Static members belongs to the class, rather th...",The `static` keyword in a function declaration...,776,9.891236,11.362051,NaN,Failed to score model on payload. Error: Provi...
2,What does the 'finally' block in Python do?,'Finally' defines a block of code to run when ...,The `finally` block in Python is used to ensur...,878,11.707354,14.610764,NaN,Failed to score model on payload. Error: Provi...
3,What is the difference between multiprocessing...,Multithreading refers to the ability of a proc...,The core difference between multiprocessing an...,862,12.697288,14.167226,NaN,Failed to score model on payload. Error: Provi...


Lets see if we can improve `basic_qa_model` by creating a new model that could perform better by changing the system prompt.

Call `mlflow.evaluate()` using the new model. Observe that the professionalism score has increased!

In [30]:
with mlflow.start_run() as run:
    system_prompt = "Answer the following question using extreme formality."
    professional_qa_model = mlflow.openai.log_model(
        model="myllm",
        task=client.chat.completions,
        artifact_path="model",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "{question}"},
        ],
    )
    results = mlflow.evaluate(
        professional_qa_model.model_uri,
        eval_df,
        model_type="question-answering",
        evaluators="default",
        extra_metrics=[professionalism_metric],
    )
print(results.metrics)

2025/06/27 22:39:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/27 22:39:40 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-6193e7db52d24b9fbbbbc38da5f5e609
2025/06/27 22:39:40 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/06/27 22:39:40 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-6193e7db52d24b9fbbbbc38da5f5e609
2025/06/27 22:39:40 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/06/27 22:39:40 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
2025/06/27 22:39:47 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
2025/06/27 22:39:50 WARNING mlflow.metrics.metric_definitions: Failed to load 'toxicity' metric (error: RuntimeError('At least one of TensorFlow 2.0 or PyTorch s

{'professionalism/v1/mean': 5.0, 'professionalism/v1/variance': 0.0, 'professionalism/v1/p90': 5.0}


In [31]:
results.tables["eval_results_table"]

,inputs,ground_truth,outputs,token_count,professionalism/v1/score,professionalism/v1/justification
0,How does useEffect() work?,The useEffect() hook tells React that your com...,"The `useEffect()` hook, a fundamental feature ...",376,5,The response is written in an excessively form...
1,What does the static keyword in a function mean?,"Static members belongs to the class, rather th...","The use of the keyword ""static"" within the con...",160,5,The response is excessively formal and respect...
2,What does the 'finally' block in Python do?,'Finally' defines a block of code to run when ...,"In the esteemed programming language Python, t...",230,5,"The response is excessively formal, using prec..."
3,What is the difference between multiprocessing...,Multithreading refers to the ability of a proc...,The distinction between multiprocessing and mu...,306,5,"The response is excessively formal, using prec..."
